<div>
    <table>
        <tr>
            <td>
                <center>
                    <h1>Premise Introduction</h1>
                     <a href="https://www.psi.ch/en/ta/people/romain-sacchi">Romain Sacchi</a> (PSI)
                    <br><br>
                    Duration: 1 hour 15 minutes.
                </center>
            </td>
        </tr>
    </div>

<div class="alert alert-info">
Note: we will be using <a href="https://docs.brightway.dev/en/latest/content/installation/index.html">Brightway 2</a>, not <a href="https://docs.brightway.dev/en/legacy/">Brightway 2.5</a>, because we would like to visualize our results in <a href="https://github.com/LCA-ActivityBrowser/activity-browser">Activity Browser</a>, which is for now only compatible with Brightway 2. But if you do not require ``activity-browser``, you can run this notebook in an environment containing ``brightway25``.
</div>

In [ ]:
pip install premise

In [7]:
from premise import __version__
__version__

(2, 3, 9)

In [2]:
from premise import *
import bw2data as bd

Link to Premise scenario dashboard: https://premisedash-6f5a0259c487.herokuapp.com/

Premise documentation: https://premise.readthedocs.io/en/latest/

Examples notebook: https://github.com/polca/premise/blob/master/examples/examples.ipynb

In [3]:
NewDatabase?

Init signature:
NewDatabase(
    scenarios: List[dict],
    source_version: str = '3.12',
    source_type: str = 'brightway',
    key: Union[bytes, str] = None,
    source_db: str = None,
    source_file_path: str = None,
    additional_inventories: List[dict] = None,
    system_model: str = 'cutoff',
    system_args: dict = None,
    use_cached_inventories: bool = True,
    use_cached_database: bool = True,
    external_scenarios: list = None,
    quiet=False,
    keep_imports_uncertainty=True,
    keep_source_db_uncertainty=False,
    gains_scenario='CLE',
    use_absolute_efficiency=False,
    biosphere_name: str = 'biosphere3',
    generate_reports: bool = True,
) -> None
Docstring:     
Class that represents a new wurst inventory database, modified according to IAM data.

:ivar source_type: the source of the ecoinvent database. Can be `brigthway` or `ecospold`.
:vartype source_type: str
:vartype source_db: str
:ivar system_model: Can be `cutoff` (default) or `consequential`.
:vart

In [3]:
bd.projects

Brightway2 projects manager with 5 objects:
	BEST502
	default
	demo
	demo2
	premise
Use `projects.report()` to get a report on all projects.

In [4]:
bd.projects.set_current("BEST502")

In [6]:
bd.databases

Databases dictionary with 4 object(s):
	carbon fiber
	ecoinvent-3.10-biosphere
	ecoinvent-3.10.1-cutoff
	ei_cutoff_3.10_image_SSP2-M_2050 2026-03-20

In [10]:
ndb = NewDatabase(
    scenarios=[
        {"model":"image", "pathway":"SSP2-M", "year":2050},
        #{"model":"remind", "pathway":"SSP2-PkBudg500", "year":2050},
    ],
    source_db="ecoinvent-3.10.1-cutoff", # <-- name of the database in the BW2 project. Must be a string.
    source_version="3.10", # <-- version of ecoinvent. Can be "3.8", "3.9" or "3.10". Must be a string.
    key='tUePmX_S5B8ieZkkM7WUU2CnO8SmShwmAeWK9x2rTFo=', # <-- decryption key - used to assess default scenarios included in `premise`
    #The key is copied from the official Premise tutorial, available at: https://github.com/polca/premise/blob/master/examples/examples.ipynb
)

premise v.(2, 3, 9)
+------------------------------------------------------------------+
| Warning                                                          |
+------------------------------------------------------------------+
| Because some of the scenarios can yield LCI databases            |
| containing net negative emission technologies (NET),             |
| it is advised to account for biogenic CO2 flows when calculating |
| Global Warming potential indicators.                             |
| `premise_gwp` provides characterization factors for such flows.  |
| It also provides factors for hydrogen emissions to air.          |
|                                                                  |
| Within your Brightway project:                                   |
| from premise_gwp import add_premise_gwp                          |
| add_premise_gwp()                                                |
+------------------------------------------------------------------+
+-------------

In [11]:
# select sector to integrate, or all sectors. for example, updating the electricity sector: ndb.update_electricity()
ndb.update("electricity")

Processing scenarios for sector 'electricity': 100%|█| 1/1 [01:08<00:0


Done!



In [12]:
import bw2io as bi
bi.bw2setup()

Creating default biosphere

Applying strategy: normalize_units
Applying strategy: drop_unspecified_subcategories
Applying strategy: ensure_categories_are_tuples
Applied 3 strategies in 0.03 seconds


100%|████████████████████████████████████████████████████████████████████████████| 4709/4709 [00:00<00:00, 9924.82it/s]


09:49:37-0700 [info     ] Vacuuming database            
Created database: biosphere3
Creating default LCIA methods



ValueError: Method ('CML v4.8 2016 no LT', 'acidification no LT', 'acidification (incl. fate, average Europe total, A&B) no LT') already exists. Use ``overwrite=True`` to overwrite existing methods

In [ ]:
ndb.write_db_to_brightway()

Write new database(s) to Brightway.
Running all checks...
Minor anomalies found: check the change report.
09:54:41-0700 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


 19%|██████████████▎                                                             | 5315/28176 [00:30<02:07, 179.65it/s]

In [23]:
ndb.write_superstructure_db_to_brightway()

ValueError: At least two scenarios are needed tocreate a super-structure database.

## Incremental Databases

The class ``IncrementalDatabase`` also implementing sectoral changes in an incremental, to easily distinguish the contribution of each sector on the results.

In [ ]:
ndb = IncrementalDatabase(
    scenarios=[
        {"model":"image", "pathway":"SSP2-RCP26", "year":2040},
    ],
    source_db="ecoinvent-3.10.-cutoff", # <-- name of the database in the BW2 project. Must be a string.
    source_version="3.10", # <-- version of ecoinvent. Can be "3.5", "3.6", "3.7" or "3.8". Must be a string.
    key="xxxxxxxxxxxxxxxxxx",
)

In [ ]:
sectors = {
    "electricity": "electricity",
    "steel": "steel",
    "others": [
        "cement",
        "cars",
        "fuels"
    ]
}

In [ ]:
ndb.update(sectors=sectors)

In [ ]:
ndb.write_increment_db_to_brightway(name="test increment", file_format="csv")

## Reports
### Scenario report
You can generate a spreadsheet report showing the main variables of the scenario you have selected to create your databases. The report is saved in your working directory. Note that this report is generated automatically when exporting a database.

In [ ]:
ndb.generate_scenario_report()

## Changes report
You can generate a spreadsheet report of the changes made to the original database. It gives an overview on:
* the datasets created
* the datasets modified
* some performance indicators
* scaling factors used to scale certain exchanges

A "Validation" tab also shows any datasets that contain values or efficiencies that may seem incorrect.

The report is saved in your working directory. It is generated automatically when you export a database.

In [ ]:
ndb.generate_change_report()

## Custom scenarios

You can also create your own scenarios by providing a data package to premise. This data package should contain the following files:
* ``datapackage.json``: a json file containing the metadata of the data package
* ``scenario_data.csv``: a csv file containing the scenario data
* ``config.yaml``: a yaml file containing the configuration of the scenario (which markets to create, etc.)
* ``lci-inventories.csv``: a csv file containing additional inventories to import

In [ ]:
import brightway2 as bw
from premise import NewDatabase
from datapackage import Package


fp = r"https://raw.githubusercontent.com/premise-community-scenarios/ammonia-prospective-scenarios/main/datapackage.json"
ammonia = Package(fp)

bw.projects.set_current("your_bw_project")

ndb = NewDatabase(
    scenarios = [
        {"model":"image", "pathway":"SSP2-Base", "year":2050, "external scenarios": [{"scenario": "Business As Usual - image", "data": ammonia}]},
        {"model":"image", "pathway":"SSP2-RCP26", "year":2030, "external scenarios": [{"scenario": "Sustainable development - image", "data": ammonia}]},
    ],        
    source_db="ecoinvent 3.10 cutoff",
    source_version="3.10",
    key='xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx',
)
    
ndb.update("external") # or ndb.update() if you want to update the database with the IAM data plus the external scenario
